In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
import yaml

from openaq import OpenAQ
from ml_project.utils import get_project_directories, setup_logger
from ml_project.data import openaq_extract_data, meteostat_extract_data
from ml_project.cleaning import clean_data
from ml_project.feature_engineering import (
    generate_features, impute_missing_data,
    one_hot_encoding,
    scaling, detect_outliers, handle_outliers
)
from ml_project.data.MeteoStatClient import MeteoStatClient

In [2]:
directory_paths_dict = get_project_directories()
env_path = directory_paths_dict["root"] / ".env"
load_dotenv(dotenv_path=env_path)

log_file_name = "ml_project.log"
log_file_path = directory_paths_dict["root"] / "logs"
logger = setup_logger(log_file_path=log_file_path / log_file_name, name="ml_project")

open_aq_api = os.getenv("OPEN_AQ_API_KEY")
meteostat_api = os.getenv("METEOSTAT_API_KEY")

open_aq_client = OpenAQ(api_key=open_aq_api)
meteo_client = MeteoStatClient(api_key=meteostat_api)

cfg_path = directory_paths_dict["configs"] / "data.yaml"
with cfg_path.open("r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

In [3]:
locations_df_raw = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["openaq"]["locations"])
sensors_metadata_df_raw = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["openaq"]["sensors_metadata"])
sensors_measurements_df_raw = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["openaq"]["sensors_measurements"])
weather_daily_df_raw = pd.read_csv(directory_paths_dict["data_raw"] / cfg["outputs"]["files"]["meteostat"]["weather_daily"])

dataframes_dict_raw = {
    "locations": locations_df_raw,
    "sensors_metadata": sensors_metadata_df_raw,
    "sensors_measurements": sensors_measurements_df_raw,
    "weather": weather_daily_df_raw
}

cleaned_data_dict = clean_data(dataframes_dict_raw)
cleaned_df = cleaned_data_dict["cleaned"]

In [4]:
cleaned_df.head()

,reading_date,bc,co,no2,o3,pm10,pm25,so2,sensor_count,tavg,tmin,tmax,prcp,snow,wspd,pres,latitude,longitude
0,2020-01-01,0.631,410.0,11.790,44.775,20.380,18.780,3.04,21,1.4,0.8,5.6,1.0,0.0,17.4,1029.3,52.237049,21.017532
1,2020-01-02,1.220,586.0,31.800,26.650,34.440,28.460,5.59,21,0.8,-1.7,3.1,0.0,0.0,13.8,1026.5,52.237049,21.017532
2,2020-01-03,1.890,819.0,38.025,20.475,40.300,35.100,9.37,21,0.3,-4.1,4.6,0.0,0.0,6.4,1020.1,52.237049,21.017532
3,2020-01-04,0.669,469.0,14.190,50.025,11.738,10.996,6.17,21,3.7,1.6,5.3,1.0,0.0,20.7,1012.3,52.237049,21.017532
4,2020-01-05,0.817,532.0,17.950,43.425,12.282,10.448,3.32,21,0.5,-2.4,2.3,1.0,1.0,17.8,1025.8,52.237049,21.017532


In [5]:
# Log missing values summary
missing_counts = cleaned_df.isna().sum()
missing_cols = missing_counts[missing_counts > 0]
if len(missing_cols) > 0:
    print("Missing values per column:")
    for col, count in missing_cols.items():
        pct = (count / len(cleaned_df)) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
else:
    print("No missing values found")

Missing values per column:
  bc: 201 (9.6%)
  co: 44 (2.1%)
  no2: 2 (0.1%)
  o3: 2 (0.1%)
  pm10: 3 (0.1%)
  pm25: 2 (0.1%)
  so2: 19 (0.9%)
  tavg: 2 (0.1%)
  tmin: 4 (0.2%)
  tmax: 2 (0.1%)
  wspd: 6 (0.3%)
  pres: 7 (0.3%)


In [6]:
# Detect outliers first (for inspection/logging)
outliers = detect_outliers(cleaned_df)

if outliers:
    print(f"Columns with outliers: {list(outliers.keys())}")
    total_outliers = sum(len(v) for v in outliers.values())
    print(f"Total outliers detected: {total_outliers}")
    for col, outlier_list in outliers.items():
        print(f"  {col}: {len(outlier_list)} outliers")
else:
    print("No outliers detected")

# Handle outliers by capping values at IQR fences
cleaned_df = handle_outliers(cleaned_df, method="cap")
print(f"Outliers capped. Shape after handling: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")

Columns with outliers: ['bc', 'co', 'no2', 'o3', 'pm10', 'pm25', 'so2', 'tmin', 'prcp', 'snow', 'wspd', 'pres']
Total outliers detected: 1029
  bc: 105 outliers
  co: 37 outliers
  no2: 51 outliers
  o3: 3 outliers
  pm10: 76 outliers
  pm25: 80 outliers
  so2: 90 outliers
  tmin: 1 outliers
  prcp: 301 outliers
  snow: 200 outliers
  wspd: 40 outliers
  pres: 45 outliers
Outliers capped. Shape after handling: 2090 rows, 18 columns


In [7]:
# Impute missing data
cleaned_df = impute_missing_data(cleaned_df)
missing_after = cleaned_df.isna().sum().sum()
print(f"Missing values after imputation: {missing_after}")
print(f"Shape after imputation: {cleaned_df.shape[0]} rows, {cleaned_df.shape[1]} columns")

Missing values after imputation: 0
Shape after imputation: 2090 rows, 18 columns


In [8]:
# Generate features
featured_df = generate_features(cleaned_df)
new_features = set(featured_df.columns) - set(cleaned_df.columns)
print(f"Generated {len(new_features)} new features")
if new_features:
    print(f"New features: {list(new_features)}")
print(f"Shape after feature generation: {featured_df.shape[0]} rows, {featured_df.shape[1]} columns")

Generated 0 new features
Shape after feature generation: 2090 rows, 18 columns


In [9]:
# One-hot encode categorical columns
encoded_df = one_hot_encoding(featured_df)
new_encoded_cols = encoded_df.shape[1] - featured_df.shape[1]
print(f"Added {new_encoded_cols} columns from one-hot encoding")
print(f"Shape after one-hot encoding: {encoded_df.shape[0]} rows, {encoded_df.shape[1]} columns")

Added 0 columns from one-hot encoding
Shape after one-hot encoding: 2090 rows, 18 columns


In [10]:
# Scale numeric features
scaled_df = scaling(encoded_df)
print(f"Shape after scaling: {scaled_df.shape[0]} rows, {scaled_df.shape[1]} columns")

Shape after scaling: 2090 rows, 18 columns
